# MedSentinel — GRPO Training Notebook
### Multi-agent RL for medical decision-making with schema drift

**Environment:** MedSentinel  
**Model:** Qwen2.5-3B-Instruct (4-bit via Unsloth)  
**Algorithm:** GRPO (Group Relative Policy Optimization) via TRL  
**Target hardware:** Kaggle T4 / Google Colab T4/A100  

This notebook:
1. Clones the MedSentinel repo and installs dependencies
2. Runs a **baseline eval** (zero-shot Qwen2.5-3B) on the test set
3. **Trains** with GRPO using MedSentinel's deterministic reward function
4. Runs a **post-training eval** to show improvement
5. Plots the **reward curve**
6. Saves the adapter to disk (optionally pushes to HuggingFace Hub)

In [ ]:
# Check GPU availability
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU detected — training will be very slow on CPU")

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install required packages
# NOTE: Unsloth install depends on your CUDA version.
# Use the correct install command from https://github.com/unslothai/unsloth

import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

# Core training stack
pip_install('trl>=0.8.0', 'transformers>=4.40.0', 'datasets>=2.18.0', 'peft>=0.10.0')
pip_install('matplotlib', 'numpy', 'openenv-core')

# Unsloth — install based on CUDA version
# For Kaggle T4 (CUDA 12.1):
try:
    pip_install('unsloth')
    print("✅ Unsloth installed")
except Exception as e:
    print(f"⚠️  Unsloth install failed: {e}")
    print("Falling back to plain HuggingFace transformers (slower)")

print("\n✅ All packages installed")

In [ ]:
import os, sys

# ── Option A: Running from inside the repo (e.g. Kaggle dataset) ──
# If you uploaded the medsentinel folder as a Kaggle dataset, set REPO_PATH:
# REPO_PATH = '/kaggle/input/medsentinel/ms_build'

# ── Option B: Clone from HuggingFace Hub ──
# Replace with your HF Space URL once deployed
# !git clone https://huggingface.co/spaces/YOUR_USERNAME/medsentinel /kaggle/working/medsentinel

# ── Option C: Unzip uploaded file ──
# If you uploaded medsentinel_final.zip as a Kaggle dataset:
# import zipfile
# with zipfile.ZipFile('/kaggle/input/medsentinel/medsentinel_final.zip', 'r') as z:
#     z.extractall('/kaggle/working/medsentinel')
# REPO_PATH = '/kaggle/working/medsentinel'

# ── Default: assume this notebook is in the repo root ──
REPO_PATH = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.exists(os.path.join(REPO_PATH, 'data', 'patient_cases.json')):
    # Try current directory
    REPO_PATH = os.getcwd()
    if not os.path.exists(os.path.join(REPO_PATH, 'data', 'patient_cases.json')):
        raise FileNotFoundError(
            f"Cannot find patient_cases.json. Set REPO_PATH to the medsentinel repo root.\n"
            f"Tried: {REPO_PATH}"
        )

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

print(f"✅ Repo found at: {REPO_PATH}")
import json
cases = json.load(open(os.path.join(REPO_PATH, 'data', 'patient_cases.json')))
print(f"✅ Dataset: {len(cases)} patient cases")

In [ ]:
# ── Training Configuration ──
# Tune these to match your hardware

BASE_MODEL     = "unsloth/qwen2.5-3b-instruct-bnb-4bit"  # matches trained adapter
STEPS          = 300      # 200-500 for T4; use 100 for quick test
BATCH_SIZE     = 2        # reduce to 1 if OOM
LR             = 5e-6
NUM_GENERATIONS = 4       # GRPO group size
MAX_SEQ_LEN    = 1024
SEED           = 42

SAVE_PATH      = os.path.join(REPO_PATH, 'checkpoints', 'medsentinel_grpo')
HUB_MODEL_ID   = None     # Set to "your-username/medsentinel-3b" to push to HF Hub
PUSH_TO_HUB    = False    # Set True to push after training

print(f"Config:")
print(f"  Model:      {BASE_MODEL}")
print(f"  Steps:      {STEPS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  LR:         {LR}")
print(f"  Save path:  {SAVE_PATH}")

In [ ]:
import sys, os
sys.path.insert(0, REPO_PATH)
sys.path.insert(0, os.path.join(REPO_PATH, 'training'))

from training.train_grpo import (
    load_grpo_dataset,
    load_model_unsloth,
    run_baseline_eval,
    build_reward_fn,
    save_reward_curve,
    format_with_system_fn,
    SYSTEM_PROMPT,
)
from env.reward_system import compute_reward
from agents.auditor_agent import audit_doctor_output

print("✅ All training functions imported")

In [ ]:
import json

print("Loading dataset...")
train_ds, test_ds, all_cases = load_grpo_dataset(
    os.path.join(REPO_PATH, 'data', 'patient_cases.json'),
    seed=SEED
)
test_cases = [json.loads(r['patient_json']) for r in test_ds]

print(f"✅ Train: {len(train_ds)} | Test: {len(test_ds)}")
print(f"   Sample prompt (first 200 chars):")
print(train_ds[0]['prompt'][:200])

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

print(f"\nLoading model: {BASE_MODEL}")
model, tokenizer, backend = load_model_unsloth(BASE_MODEL, MAX_SEQ_LEN)
print(f"✅ Model loaded via {backend}")

In [ ]:
print("=" * 50)
print("BASELINE EVALUATION (zero-shot, before training)")
print("=" * 50)

baseline = run_baseline_eval(
    model, tokenizer, test_cases,
    n_cases=min(20, len(test_cases)),
    device=DEVICE
)

print(f"\nBaseline Results:")
print(f"  Avg Reward : {baseline['avg_reward']:.3f}")
print(f"  Accuracy   : {baseline['accuracy_pct']:.1f}%")
print(f"  Safety     : {baseline['safety_pct']:.1f}%")
print(f"  N evaluated: {baseline['n_evaluated']}")

In [ ]:
from trl import GRPOConfig, GRPOTrainer
import time

# Build reward function
patients_by_id = {c['patient_id']: c for c in all_cases}
reward_fn = build_reward_fn(patients_by_id)

# Apply chat template to dataset
def fmt(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": example["prompt"]},
    ]
    return {
        "prompt":        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
        "patient_json":  example["patient_json"],
        "drift_occurred":example.get("drift_occurred", "False"),
    }

print("Formatting dataset with chat template...")
train_ds_formatted = train_ds.map(fmt)

# GRPO config
grpo_config = GRPOConfig(
    output_dir=SAVE_PATH,
    num_train_epochs=1,
    max_steps=STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=max(1, 8 // BATCH_SIZE),
    learning_rate=LR,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_SEQ_LEN // 2,
    max_completion_length=512,
    temperature=0.9,
    logging_steps=10,
    save_steps=100,
    seed=SEED,
    report_to="none",
    remove_unused_columns=False,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=grpo_config,
    train_dataset=train_ds_formatted,
    processing_class=tokenizer,
)

# Capture reward log
reward_log = []
_orig_log = trainer.log
def _patched_log(logs, *a, **kw):
    if "rewards/mean" in logs: reward_log.append(logs["rewards/mean"])
    elif "reward" in logs:     reward_log.append(logs["reward"])
    _orig_log(logs, *a, **kw)
trainer.log = _patched_log

print(f"\nStarting GRPO training for {STEPS} steps...")
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f"\n✅ Training complete in {elapsed/60:.1f} minutes")

In [ ]:
import os
os.makedirs(SAVE_PATH, exist_ok=True)
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Model saved to {SAVE_PATH}")

In [ ]:
if backend == "unsloth":
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)

print("=" * 50)
print("POST-TRAINING EVALUATION")
print("=" * 50)

post = run_baseline_eval(
    model, tokenizer, test_cases,
    n_cases=min(20, len(test_cases)),
    device=DEVICE
)

print(f"\nResults:")
print(f"  {'Metric':<20} {'Before':>10} {'After':>10} {'Delta':>10}")
print(f"  {'-'*50}")
print(f"  {'Avg Reward':<20} {baseline['avg_reward']:>10.3f} {post['avg_reward']:>10.3f} {post['avg_reward']-baseline['avg_reward']:>+10.3f}")
print(f"  {'Accuracy %':<20} {baseline['accuracy_pct']:>10.1f} {post['accuracy_pct']:>10.1f} {post['accuracy_pct']-baseline['accuracy_pct']:>+10.1f}")
print(f"  {'Safety %':<20} {baseline['safety_pct']:>10.1f} {post['safety_pct']:>10.1f} {post['safety_pct']-baseline['safety_pct']:>+10.1f}")

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

if reward_log:
    steps_x = list(range(1, len(reward_log) + 1))
    window = min(10, len(reward_log))
    smoothed = np.convolve(reward_log, np.ones(window)/window, mode='valid')

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(steps_x, reward_log, alpha=0.3, color='steelblue', label='Raw reward')
    ax.plot(steps_x[window-1:], smoothed, color='steelblue', linewidth=2.5, label=f'Smoothed (w={window})')
    ax.axhline(y=baseline['avg_reward'], color='red', linestyle='--', alpha=0.7,
               label=f'Baseline ({baseline["avg_reward"]:.3f})')
    ax.axhline(y=post['avg_reward'], color='green', linestyle='--', alpha=0.7,
               label=f'Post-train ({post["avg_reward"]:.3f})')
    ax.axhline(y=0, color='gray', linestyle=':', alpha=0.4)
    ax.set_xlabel('Training Step', fontsize=12)
    ax.set_ylabel('Average Reward', fontsize=12)
    ax.set_title('MedSentinel GRPO Training — Reward Curve', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    fig.tight_layout()

    curve_path = os.path.join(REPO_ROOT, 'dashboard', 'static', 'reward_curve.png') if 'REPO_ROOT' in dir() else os.path.join(SAVE_PATH, 'reward_curve.png')
    os.makedirs(os.path.dirname(curve_path), exist_ok=True)
    fig.savefig(curve_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ Reward curve saved to {curve_path}")
else:
    print("No reward log captured — check trainer logging_steps config")

In [ ]:
import json

summary = {
    "baseline":        baseline,
    "post_training":   post,
    "training_steps":  STEPS,
    "base_model":      BASE_MODEL,
    "reward_log":      reward_log,
    "elapsed_minutes": elapsed / 60,
}
summary_path = os.path.join(SAVE_PATH, 'training_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"✅ Summary saved to {summary_path}")
print(json.dumps({k: v for k, v in summary.items() if k != 'reward_log'}, indent=2))

In [ ]:
# Push trained model to HuggingFace Hub
# Set HUB_MODEL_ID = "your-username/medsentinel-3b" and PUSH_TO_HUB = True above

if PUSH_TO_HUB and HUB_MODEL_ID:
    print(f"Pushing to HuggingFace Hub: {HUB_MODEL_ID}")
    model.push_to_hub(HUB_MODEL_ID)
    tokenizer.push_to_hub(HUB_MODEL_ID)
    print(f"✅ Model live at https://huggingface.co/{HUB_MODEL_ID}")
else:
    print("Skipping HF Hub push (set PUSH_TO_HUB=True and HUB_MODEL_ID to enable)")